# Лабораторна робота № 5
## Географічні дані OpenStreetMap: від тегів до маршруту

**Інтелектуальний аналіз та візуалізація даних** · 4 години · 4 бали

---

### Постановка

Ви — аналітик у туристичному застосунку. Завдання: для заданого міста
**відібрати десять найвартісніших історико-культурних об'єктів** і
**спланувати піший маршрут**, який починається на залізничному вокзалі,
обходить усі десять і повертається на вокзал.

Слово «найвартісніші» в даних не записане. Критерій ви побудуєте самі —
і захистите його.

Потім те саме для **тридцяти** об'єктів. Результат цієї, більшої задачі
стане базою порівняння в **лабораторній роботі № 8**, де ту саму задачу
розв'язуватимуть генетичним та мурашиним алгоритмами.

### Що здаєте

1. Цей ноутбук із виконаними комірками (усі виводи збережені).
2. Файл `lr5_результати.json` — результати для ЛР8.
3. Карту `lr5_маршрут.html`.
4. Звіт у кінці ноутбука: відповіді на сім питань.

### Що оцінюється

| критерій | 0 | 2 | 3 | 4 |
|---|---|---|---|---|
| Відбір об'єктів | теги взято навмання | фільтр працює | фільтр обґрунтовано | показано, що відкинуто і чому |
| Критерій рейтингу | немає | є формула | ваги пояснено | порівняно з альтернативним критерієм |
| Маршрут | не будується | будується | перевірено коректність | пояснено різницю методів числом |
| Карта | немає | точки на карті | маршрут по ребрах графа | видно різницю з прямими відрізками |
| Звіт | немає | формально | з числами | з висновком, якого не було в завданні |

---
## 0. Ваш варіант

Впишіть прізвище та номер у списку групи. Місто підтягнеться само.

In [1]:
STUDENT_NAME = 'Лящук Соломія'
LIST_NUMBER  = 6

CITIES = [
    'Львів', 'Київ', 'Харків', 'Одеса', 'Дніпро',
    'Чернівці', 'Івано-Франківськ', 'Тернопіль', 'Ужгород', 'Луцьк',
    'Рівне', 'Хмельницький', 'Вінниця', 'Житомир', 'Чернігів',
    'Черкаси', 'Полтава', 'Суми', 'Кропивницький', 'Запоріжжя',
    'Миколаїв', 'Кривий Ріг', 'Кам\u2019янець-Подільський', 'Мукачево',
    'Коломия', 'Дрогобич', 'Біла Церква', 'Ніжин', 'Остріг', 'Умань',
]

assert STUDENT_NAME and LIST_NUMBER > 0, 'впишіть прізвище та номер у списку'
CITY = CITIES[(LIST_NUMBER - 1) % len(CITIES)] + ', Україна'
print('варіант %d: %s' % (LIST_NUMBER, CITY))

варіант 6: Чернівці, Україна


> Якщо викладач призначив вам інше місто — просто перепишіть `CITY` вручну
> і зазначте це у звіті.

In [2]:
import importlib.util, subprocess, sys
REQUIRED = ['osmnx', 'geopandas', 'shapely', 'pyproj', 'pyarrow',
            'folium', 'mapclassify', 'h3', 'networkx']
missing = [p for p in REQUIRED if importlib.util.find_spec(p) is None]
if missing:
    print('встановлюю:', ', '.join(missing), '...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('усі пакети на місці')

усі пакети на місці


In [3]:
import warnings; warnings.filterwarnings('ignore')
import json, time, os, collections
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import matplotlib.pyplot as plt
import folium
from xyzservices import providers as xyz

BASEMAP = xyz.Esri.WorldGrayCanvas   # CARTO з 2026 р. вимагає ключ API
pd.set_option('display.width', 200)
print('geopandas', gpd.__version__)

geopandas 1.2.0


---
## 1. Два способи дістати дані

Робоча програма вимагає порівняти два способи доступу **за обсягом переданих
даних і за часом**:

* **живий запит** — `osmnx` питає Overpass API просто зараз;
* **знімок** — ті самі дані, збережені у файли GeoParquet, читаються з диска.

Нижче — живий запит. Запустіть його **один раз**: він збереже теку зі знімком,
і всі наступні запуски читатимуть з неї.

> Overpass — безкоштовний сервіс і обмежує частоту. Якщо запит не пройшов —
> зачекайте хвилину й повторіть. Якщо не проходить взагалі, візьміть
> резервний знімок у викладача і вкажіть це у звіті.

In [4]:
import osmnx as ox

OUT = 'osm_' + CITY.split(',')[0].strip().lower().replace(' ', '_')
os.makedirs(OUT, exist_ok=True)
ox.settings.use_cache = True
ox.settings.cache_folder = os.path.join(OUT, 'cache')
ox.settings.requests_timeout = 300
ox.settings.overpass_rate_limit = True

TOURISM_OK = ['museum', 'attraction', 'gallery', 'artwork', 'viewpoint']

def cache_bytes():
    import pathlib
    p = pathlib.Path(ox.settings.cache_folder)
    return sum(f.stat().st_size for f in p.glob('*.json')) if p.exists() else 0

print('тека знімка:', OUT)

тека знімка: osm_чернівці


### Дві налаштовані дрібниці, без яких запит не пройде

Це не формальності — кожна з них ламає збір по-своєму, і жодна не схожа
на свою справжню причину.

**Представитись.** Overpass відповідає **HTTP 406** клієнтові, який прийшов
із типовим заголовком `python-requests`. `osmnx` надсилає свій `User-Agent`
сам, але якщо ви звертаєтесь до сервісу напряму — надсилайте теж.
Рядок має бути **лише латиницею**: HTTP-заголовки передаються в latin-1,
і кирилиця кладе запит помилкою `UnicodeEncodeError` ще до відправлення.

**Обрати машину.** За іменем `overpass-api.de` стоїть кілька серверів, і
частина з них відмовляє у з’єднанні. `osmnx` закріплює одну адресу на
всю сесію — навмисно, щоб не порушувати облік слотів. Якщо закріпиться та,
що відмовляє, впаде весь збір. Тому адресу обираємо самі, вже перевірену.

In [5]:
import socket

USER_AGENT = ('LvivPolytechnic-SK26-coursework/1.0 '
              '(https://lpnu.ua; student lab assignment)')
assert USER_AGENT.isascii(), 'HTTP-заголовки — latin-1, кирилиця тут не працює'
ox.settings.http_user_agent = USER_AGENT


def pin_working_ip(hostname, port=443, timeout=8):
    """Закріпити для цього імені адресу, яка справді відповідає."""
    try:
        infos = socket.getaddrinfo(hostname, port, socket.AF_INET, socket.SOCK_STREAM)
    except Exception as e:
        print('не вдалось резолвити %s: %s' % (hostname, type(e).__name__))
        return None
    ips = sorted({i[4][0] for i in infos})
    good = []
    for ip in ips:
        s = socket.socket(); s.settimeout(timeout)
        try:
            s.connect((ip, port)); good.append(ip)
        except Exception:
            pass
        finally:
            s.close()
    print('%s: машин %d, відповідають %d %s' % (hostname, len(ips), len(good), good))
    if not good:
        return None
    original = socket.gethostbyname
    socket.gethostbyname = lambda h: good[0] if h == hostname else original(h)
    print('закріплено', good[0])
    return good[0]


pin_working_ip('overpass-api.de')

overpass-api.de: машин 2, відповідають 2 ['162.55.144.139', '65.109.112.52']
закріплено 162.55.144.139


'162.55.144.139'

> Якщо жодна машина не відповіла — виконайте комірку ще раз за хвилину,
> або візьміть у викладача резервний знімок і зазначте це у звіті.
> Інші дзеркала перелічені на сторінці
> [Overpass API](https://wiki.openstreetmap.org/wiki/Overpass_API);
> адресу можна підставити у `ox.settings.overpass_url`.

### Завдання 1.1

Допишіть функцію `fetch`: вона має зберегти шар у `.parquet`, а якщо файл
уже є — прочитати його, **не звертаючись до мережі**. Поверніть також,
скільки МБ пішло мережею і скільки секунд це зайняло.

Підказка щодо `clean`: у OSM одна колонка може містити і число, і список
(ребро, склеєне з кількох шляхів). Parquet вимагає один тип на колонку,
тому неоднорідну колонку треба звести до тексту **цілком**.

In [6]:
def clean(gdf):
    g = gdf.reset_index()
    for c in g.columns:
        if c == 'geometry':
            continue
        boxed = g[c].map(lambda v: isinstance(v, (list, dict, set, tuple))).any()
        if boxed or g[c].dtype == object:
            g[c] = g[c].map(
                lambda v: None if v is None or (isinstance(v, float) and pd.isna(v))
                else (json.dumps(v, ensure_ascii=False)
                      if isinstance(v, (list, dict, set, tuple)) else str(v))
            ).astype('string')
    return g


COST = {}

def fetch(name, getter):
    """Повертає шар. Якщо файл уже збережено — читає з диска."""
    path = os.path.join(OUT, name + '.parquet')
    if os.path.exists(path) and os.path.getsize(path) > 0:
        t0 = time.time()
        g = gpd.read_parquet(path)
        COST[name] = {'спосіб': 'знімок', 'записів': len(g),
                      'передано_МБ': 0.0,
                      'файл_МБ': round(os.path.getsize(path) / 1e6, 3),
                      'секунд': round(time.time() - t0, 3)}
        return g

    # ----- живий запит -----
    t0, b0 = time.time(), cache_bytes()      # час і розмір кешу ДО запиту
    g = clean(getter())                      # питаємо Overpass і чистимо таблицю
    g.to_parquet(path, index=False)          # зберігаємо знімок
    g = gpd.read_parquet(path)
    COST[name] = {'спосіб': 'живий запит', 'записів': len(g),
                  'передано_МБ': round((cache_bytes() - b0) / 1e6, 3),
                  'файл_МБ': round(os.path.getsize(path) / 1e6, 3),
                  'секунд': round(time.time() - t0, 1)}
    return g

In [7]:
boundary = fetch('boundary', lambda: ox.geocode_to_gdf(CITY))
poly = boundary.geometry.iloc[0]
print('межа міста отримана, площа %.1f км²'
      % (boundary.to_crs(32635).area.iloc[0] / 1e6))

межа міста отримана, площа 151.4 км²


In [8]:
heritage = fetch('heritage', lambda: ox.features_from_polygon(
    poly, {'historic': True, 'tourism': TOURISM_OK}))
stations = fetch('stations', lambda: ox.features_from_polygon(
    poly, {'railway': ['station', 'halt'], 'building': ['train_station']}))
amenities = fetch('amenities', lambda: ox.features_from_polygon(
    poly, {'amenity': ['cafe', 'restaurant', 'drinking_water', 'toilets']}))

# Мережа приходить двома таблицями одразу, тому їй окрема гілка.
NODES_P = os.path.join(OUT, 'road_nodes.parquet')
EDGES_P = os.path.join(OUT, 'road_edges.parquet')

if os.path.exists(NODES_P) and os.path.exists(EDGES_P):
    t0 = time.time()
    nodes, edges = gpd.read_parquet(NODES_P), gpd.read_parquet(EDGES_P)
    COST['мережа walk'] = {'спосіб': 'знімок', 'записів': len(nodes) + len(edges),
                           'передано_МБ': 0.0,
                           'файл_МБ': round((os.path.getsize(NODES_P)
                                             + os.path.getsize(EDGES_P)) / 1e6, 3),
                           'секунд': round(time.time() - t0, 3)}
else:
    t0, b0 = time.time(), cache_bytes()
    G_raw = ox.graph_from_polygon(poly, network_type='walk')
    nodes, edges = ox.convert.graph_to_gdfs(G_raw)
    clean(nodes).to_parquet(NODES_P, index=False)
    clean(edges).to_parquet(EDGES_P, index=False)
    nodes, edges = gpd.read_parquet(NODES_P), gpd.read_parquet(EDGES_P)
    COST['мережа walk'] = {'спосіб': 'живий запит', 'записів': len(nodes) + len(edges),
                           'передано_МБ': round((cache_bytes() - b0) / 1e6, 3),
                           'файл_МБ': round((os.path.getsize(NODES_P)
                                             + os.path.getsize(EDGES_P)) / 1e6, 3),
                           'секунд': round(time.time() - t0, 1)}

print('вузлів %d | ребер %d' % (len(nodes), len(edges)))

вузлів 14240 | ребер 37950


### Завдання 1.2

Зведіть `COST` у таблицю і дайте відповідь у звіті: **у скільки разів
читання знімка швидше за живий запит** і **скільки мегабайтів ви не передали
мережею** при повторному запуску.

Контрольна ознака правильності: при **другому** запуску ноутбука всі рядки
мають мати спосіб «знімок», а `передано_МБ` — нуль.

In [9]:
snap_table = pd.DataFrame(COST).T
display(snap_table)

# Скільки коштував ЖИВИЙ запит. Беремо live_cost.json (числа з журналу першого збору),
# а якщо його немає — manifest.json, який записав fetch_city_osm.py.
layers = ['boundary', 'heritage', 'stations', 'amenities', 'road_nodes', 'road_edges']
live = None
for fname in ['live_cost.json', 'manifest.json']:
    fp = os.path.join(OUT, fname)
    if os.path.exists(fp):
        live = json.load(open(fp, encoding='utf-8')); print('живі числа з', fname); break

if live:
    live_sec = sum(live[k]['секунд'] for k in layers if k in live)
    live_mb = sum(live[k]['передано_МБ'] for k in layers if k in live)
    snap_sec = snap_table['секунд'].sum()
    print('живий запит : %.1f с, передано %.2f МБ' % (live_sec, live_mb))
    print('знімок      : %.3f с, передано 0 МБ' % snap_sec)
    print('знімок швидший у %.0f раз(ів); мережею не передано %.2f МБ' % (live_sec / snap_sec, live_mb))

,спосіб,записів,передано_МБ,файл_МБ,секунд
boundary,знімок,1,0.0,0.034,0.262
heritage,знімок,93,0.0,0.046,0.018
stations,знімок,15,0.0,0.036,0.011
amenities,знімок,453,0.0,0.078,0.013
мережа walk,знімок,52190,0.0,2.947,0.037


живі числа з live_cost.json
живий запит : 318.3 с, передано 7.33 МБ
знімок      : 0.341 с, передано 0 МБ
знімок швидший у 933 раз(ів); мережею не передано 7.33 МБ


> **Чому пішохідна мережа, а не автомобільна.** У старому місті музеї стоять
> за сотню метрів один від одного, а автомобільних вузлів там одиниці. Кілька
> пам'яток сядуть на **один** вузол графа, і матриця віддалей скаже, що між
> ними нуль метрів. Задача розв'яжеться, відповідь буде неправильною, і
> повідомлення про помилку не з'явиться. У кроці 4 ви це перевірите явно.

---
## 2. Хто претендує на звання пам'ятки

У OpenStreetMap немає тега «визначна пам'ятка». Є два сімейства:

* `historic=*` — `castle`, `church`, `monument`, `memorial`, `ruins`, `tower`, `yes`…
* `tourism=*` — `museum`, `gallery`, `attraction`, `artwork`, `viewpoint`

Перетин неповний: музей у сучасній будівлі має лише `tourism`, руїни фортеці —
лише `historic`. Тому беремо **об'єднання**.

📖 <https://wiki.openstreetmap.org/wiki/Map_Features>

### Завдання 2.1

Відберіть кандидатів і **назвіть у звіті, скільки об'єктів ви відкинули й чому**.

In [10]:
# дрібні знаки в OSM мають historic=*, але пам'ятками не є
MINOR = ['boundary_stone', 'wayside_cross', 'wayside_shrine', 'milestone', 'survey_point']

has_tag  = heritage['tourism'].isin(TOURISM_OK) | heritage['historic'].notna()  # об'єднання двох сімейств
has_name = heritage['name'].notna()                                             # без назви об'єкт у маршруті не покажеш
is_minor = heritage['historic'].isin(MINOR)

cand = heritage[has_tag & has_name & ~is_minor].reset_index(drop=True)
cand['key'] = cand['element'] + cand['id'].astype(str)   # унікальний код об'єкта

print('усього у шарі heritage :', len(heritage))
print('підходять за тегами    :', int(has_tag.sum()))
print('з них без name         :', int((has_tag & ~has_name).sum()))
print('з них дрібні знаки     :', int((has_tag & has_name & is_minor).sum()))
print('лишилось кандидатів    :', len(cand))
print('відкинуто всього       :', len(heritage) - len(cand))

усього у шарі heritage : 93
підходять за тегами    : 93
з них без name         : 21
з них дрібні знаки     : 1
лишилось кандидатів    : 71
відкинуто всього       : 22


In [11]:
assert 'cand' in dir(), 'спершу виконайте завдання 2.1'
print('=== historic ===');  print(cand['historic'].value_counts().head(10).to_string())
print()
print('=== tourism ===');   print(cand['tourism'].value_counts().head(10).to_string())
print()
print('типи геометрії:', cand.geometry.geom_type.value_counts().to_dict())

=== historic ===
historic
memorial    32
monument     6
castle       1
church       1

=== tourism ===
tourism
museum        12
attraction    12
artwork        4
viewpoint      3
gallery        1

типи геометрії: {'Point': 68, 'Polygon': 3}


> **Перевірте себе.** Серед типів геометрії мають бути і `Point`, і `Polygon`.
> Якщо самі лише полігони — ви відфільтрували зайве: пам'ятники, меморіали та
> оглядові точки в OSM є **вузлами**, у них немає будівлі.

### Якщо кандидатів менше тридцяти

Невелике місто може дати 15–20 об'єктів. Тоді:

1. розширте перелік `tourism` (наприклад, додайте `theatre`, `place_of_worship`);
2. або зменште `N_LARGE` до того, що є;

і **обов'язково зазначте це у звіті** разом із числом.

---
## 3. Критерій «найкращі»

Ключовий крок роботи. Стовпця «цінність» у даних немає. Його треба
сконструювати і захистити.

Сигнали, які є в OSM:

| сигнал | що означає |
|---|---|
| `wikidata` / `wikipedia` | про об'єкт є стаття — хтось сторонній визнав його вартим опису |
| кількість колонок `name:*` | назву переклали N мовами — перекладають те, що шукають іноземці |
| `tourism=museum/gallery` | призначений для відвідування |
| `historic` заповнений | має історичну природу |
| площа полігона | розмір — **найслабший сигнал**: великим буває і склад |

### Завдання 3.1

Побудуйте числовий рейтинг. Вимоги:

* використати **щонайменше три** різні сигнали;
* звести різнорідні величини до спільної шкали (підказка: `rank(pct=True)`);
* **пояснити у звіті, чому саме такі ваги**.

In [ ]:
metric = cand.to_crs(32635)
cand['area_m2'] = np.where(metric.geometry.geom_type.isin(['Polygon', 'MultiPolygon']),
                           metric.area.values, 0.0)

bpath = os.path.join(OUT, 'buildings.parquet')
if os.path.exists(bpath):
    bl = gpd.read_parquet(bpath).to_crs(32635)
    bl = bl[bl.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])].copy()
    bl['b_area'] = bl.area
    pts_c = gpd.GeoDataFrame({'key': cand['key']},
                             geometry=metric.geometry.representative_point().values, crs=32635)
    joined = gpd.sjoin(pts_c, bl[['b_area', 'geometry']], how='left', predicate='within')
    area_b = joined.groupby('key')['b_area'].max()    
    cand['area_m2'] = cand['key'].map(area_b).fillna(cand['area_m2'])
print('об\'єктів із площею > 0:', int((cand['area_m2'] > 0).sum()), 'з', len(cand))

for col in ['wikidata', 'wikipedia']:       
    if col not in cand.columns:
        cand[col] = None

lang_cols = [c for c in cand.columns if c.startswith('name:') and 'etymology' not in c and c != 'name:uk']

cand['wiki']     = (cand['wikidata'].notna() | cand['wikipedia'].notna()).astype(int)  
cand['n_lang']   = cand[lang_cols].notna().sum(axis=1)                                 
cand['is_visit'] = cand['tourism'].isin(['museum', 'gallery']).astype(int)             
cand['is_hist']  = cand['historic'].notna().astype(int)                               

cand['score'] = (0.35 * cand['wiki']
               + 0.30 * cand['n_lang'].rank(pct=True)
               + 0.15 * cand['is_visit']
               + 0.10 * cand['is_hist']
               + 0.10 * cand['area_m2'].rank(pct=True))

print('кандидатів   :', len(cand))
print('мають wiki   :', int(cand['wiki'].sum()))
print('мають 1+ мову:', int((cand['n_lang'] > 0).sum()))

об'єктів із площею > 0: 19 з 71
кандидатів   : 71
мають wiki   : 7
мають 1+ мову: 19


### Завдання 3.2 — обов'язкове порівняння

Побудуйте **другий** рейтинг за найпростішим критерієм — площею будівлі —
і порахуйте, скільки об'єктів збігається в перших десяти.

Це головне питання роботи: обидва списки отримані з тих самих даних, тим
самим кодом, без жодної помилки. Якщо вони різні — поясніть у звіті, що це
означає для слова «найкращі».

In [ ]:
N_SMALL, N_LARGE = 10, 30
N_LARGE = min(N_LARGE, len(cand))    

by_score = cand.sort_values(['score', 'name'], ascending=[False, True], kind='stable').reset_index(drop=True)
top_score = by_score.head(N_LARGE).reset_index(drop=True)
top10     = top_score.head(N_SMALL)
top_naive = (cand.sort_values(['area_m2', 'name'], ascending=[False, True], kind='stable')
                 .head(N_SMALL).reset_index(drop=True))
overlap   = len(set(top10['key']) & set(top_naive['key']))

print('перетин: %d з %d' % (overlap, N_SMALL))
display(pd.DataFrame({'за вашим критерієм': list(top10['name']),
                      'за площею': list(top_naive['name'])}))

перетин: 4 з 10


,за вашим критерієм,за площею
0,Костел Воздвиження Святого Христа,Музей Авіації та Космонавтики
1,Музей Володимира Івасюка,Німецький будинок
2,Музей пожежно-рятувальної справи,Німецький будинок
3,Музей Буковинської Діаспори,Художній музей
4,Franz Josef,Костел Воздвиження Святого Христа
5,Пам'ятник Сидору Воробкевичу,"Першому митрополиту Буковини і Далмації, Євген..."
6,Турецька криниця,Музей етнографії та стародавньої історії
7,Художній музей,Музей історії та культури євреїв Буковини
8,Музей етнографії та стародавньої історії,Колишня єврейська лікарня
9,Музей Авіації та Космонавтики,Природничий музей ЧНУ


Обидва списки зроблені з тих самих даних і тим самим кодом, без жодної помилки, а різні. Значить, слово «найкращі» не лежить у даних, його визначає критерій, який обрав аналітик. Площа міряє розмір, мій рейтинг міряє впізнаваність і призначення для відвідувачів. 

### Завдання 3.3 — дублікати

OSM наповнюють різні люди. Той самий музей трапляється двічі під різними
назвами, а вежа буває окремим об'єктом усередині палацу. Знайдіть у `top_score`
пари, що стоять ближче **60 метрів** одна до одної, і вирішіть, що з ними робити.

Рішення (лишити / злити / видалити) опишіть у звіті.

In [ ]:
def close_pairs(gdf, limit_m=60):
    p = gdf.to_crs(32635).geometry.representative_point()
    XY = np.c_[p.x.values, p.y.values]
    D = np.sqrt(((XY[:, None, :] - XY[None, :, :]) ** 2).sum(-1))
    np.fill_diagonal(D, np.inf)
    out = [(i, j, D[i, j]) for i in range(len(XY)) for j in range(i + 1, len(XY))
           if D[i, j] < limit_m]
    return sorted(out, key=lambda t: t[2])

pairs = close_pairs(top_score)
print('пар ближче 60 м у top_score:', len(pairs))
for i, j, d in pairs:
    print('  %5.1f м | #%d %s  <->  #%d %s'
          % (d, i + 1, top_score['name'][i], j + 1, top_score['name'][j]))

def dedupe(df, limit_m=60):
    pts = df.to_crs(32635).geometry.representative_point()
    keep = []
    for i in range(len(df)):
        if all(pts.iloc[i].distance(pts.iloc[k]) >= limit_m for k in keep):
            keep.append(i)
    return df.iloc[keep].reset_index(drop=True)

cand_clean = dedupe(by_score)
removed = len(cand) - len(cand_clean)

old_keys = set(top_score['key'])
top_score = cand_clean.head(N_LARGE).reset_index(drop=True)   
top10     = top_score.head(N_SMALL)
overlap   = len(set(top10['key']) & set(top_naive['key']))

print()
print('видалено дублікатів у всіх кандидатах:', removed)
print('змінилось у top_score                 :', len(set(top_score['key']) - old_keys))
print('перетин із площею після очищення      : %d з %d' % (overlap, N_SMALL))

пар ближче 60 м у top_score: 3
    4.9 м | #21 Німецький будинок  <->  #22 Німецький будинок
   45.0 м | #23 Steinbarg  <->  #25 Вуличне мистецтво
   54.1 м | #1 Костел Воздвиження Святого Христа  <->  #15 600 років моєму рідному місту

видалено дублікатів у всіх кандидатах: 12
змінилось у top_score                 : 3
перетин із площею після очищення      : 4 з 10


---
## 4. Старт: вокзал, і прив'язка точок до графа

Ідея етапу. Маршрут рахується по графі доріг, а об'єкти стоять у довільних точках. Тому кожну точку треба прив'язати до найближчого вузла (перехрестя).

In [15]:
print('станції у шарі:', len(stations))
print(stations[['name', 'railway', 'building']].dropna(how='all').head(12).to_string())

станції у шарі: 15
                             name  railway       building
0               Чернівці-Північна  station            NaN
1                        Чернівці  station            NaN
2                     Калинівська  station            NaN
3                         Садгора  station            NaN
4               Чернівці-Південна  station            NaN
5                  Автостанція №4      NaN            NaN
6                         Садгора      NaN            NaN
7                         Кемпінг      NaN            NaN
8    Автовокзал "Чернівці" (АС-1)      NaN            yes
9                             NaN      NaN  train_station
10  Залізничний вокзал "Чернівці"      NaN  train_station
11                            NaN      NaN  train_station


In [ ]:
city_name = CITY.split(',')[0].strip()
st = stations[stations['name'].notna()]

exact   = st[st['name'] == city_name]                                  
by_name = st[st['name'].str.contains(city_name, case=False, regex=False)]
station = (exact if len(exact) else by_name if len(by_name) else st).head(1).reset_index(drop=True)

print('старт:', station['name'].iloc[0])
print('тип геометрії:', station.geometry.geom_type.iloc[0])

старт: Чернівці
тип геометрії: Point


In [17]:
G = nx.DiGraph()
for t in nodes.itertuples():
    G.add_node(t.osmid, x=t.x, y=t.y)
for t in edges.itertuples():
    L = float(t.length)
    if G.has_edge(t.u, t.v):
        G[t.u][t.v]['length'] = min(G[t.u][t.v]['length'], L)
    else:
        G.add_edge(t.u, t.v, length=L)
print('граф: %d вузлів, %d ребер' % (G.number_of_nodes(), G.number_of_edges()))

NODES_M = nodes.to_crs(32635)
NODE_XY = np.c_[NODES_M.geometry.x.values, NODES_M.geometry.y.values]
NODE_ID = nodes['osmid'].values

def snap(points_gdf):
    """Найближчий вузол графа до кожної точки. Рахуємо в метричній CRS."""
    pmet = points_gdf.to_crs(32635)
    ids, dist = [], []
    for p in pmet.geometry:
        d2 = (NODE_XY[:, 0] - p.x) ** 2 + (NODE_XY[:, 1] - p.y) ** 2
        k = int(np.argmin(d2))
        ids.append(NODE_ID[k]); dist.append(float(np.sqrt(d2[k])))
    return np.array(ids), np.array(dist)

def make_points(top):
    names = ['СТАРТ: ' + str(station['name'].iloc[0])] + list(top['name'])
    geoms = [station.geometry.iloc[0].representative_point()] + \
            [g.representative_point() for g in top.geometry]
    return gpd.GeoDataFrame({'name': names}, geometry=geoms, crs='EPSG:4326')

граф: 14240 вузлів, 37456 ребер


### Завдання 4.1 — обов'язкова перевірка

Напишіть `check_snap`, яка повідомляє:

* скільки точок і скільки **різних** вузлів вони зайняли;
* медіану й максимум відстані «точка → вузол»;
* якщо вузлів менше, ніж точок — **назви точок, що злиплись**.

**Контрольна ознака:** на пішохідній мережі різних вузлів має бути **стільки ж,
скільки точок**. Якщо менше — ви взяли автомобільну мережу або місто надто мале;
зафіксуйте це у звіті, бо далі такі пари отримають нульову віддаль.

In [32]:
def check_snap(points, ids, dist, label):
    n, u = len(points), len(set(ids))
    print('%s: точок %d, різних вузлів %d' % (label, n, u))
    print('   відстань точка -> вузол: медіана %.1f м, максимум %.1f м'
          % (np.median(dist), np.max(dist)))
    if u < n:
        print('   УВАГА: точки злиплись на одному вузлі:')
        for node, cnt in collections.Counter(ids).items():
            if cnt > 1:
                names = [str(points['name'].iloc[i]) for i in range(n) if ids[i] == node]
                print('     вузол %s: %s' % (node, ' | '.join(names)))
    else:
        print('   усі точки на різних вузлах: добре')

---
## 5. Матриця віддалей

Дві матриці: **пряма лінія** і **дорогою**. Їхнє відношення — коефіцієнт
обходу — показує, наскільки пряма лінія бреше.

### Завдання 5.1

Ідея етапу. Задача комівояжера оперує віддалями між усіма парами точок. Будуємо дві матриці: пряму (по лінійці) і дорогою (по вулицях). Їхнє відношення, коефіцієнт обходу, показує, наскільки пряма лінія «бреше».

In [19]:
def road_matrix(ids):
    """Матриця найкоротших віддалей графом між заданими вузлами."""
    k = len(ids)
    D = np.full((k, k), np.inf)
    for i, s in enumerate(ids):
        lengths = nx.single_source_dijkstra_path_length(G, s, weight='length')
        for j, t in enumerate(ids):
            D[i, j] = lengths.get(t, np.inf)
    np.fill_diagonal(D, 0.0)
    return D

def straight_matrix(points):
    """Матриця прямих віддалей у метрах (теорема Піфагора в метричній CRS)."""
    p = points.to_crs(32635)
    XY = np.c_[p.geometry.x.values, p.geometry.y.values]
    return np.sqrt(((XY[:, None, :] - XY[None, :, :]) ** 2).sum(-1))

def detour_factor(D, E):
    """Медіана відношення дорога / пряма по парах, де обидві величини скінченні й > 0."""
    ok = np.isfinite(D) & (E > 0)
    return float(np.median(D[ok] / E[ok]))

**Контрольні ознаки — звіртесь, не підганяючи:**

* коефіцієнт обходу для пішохідної мережі у місті зазвичай **1,2–1,6**;
* якщо у вас рівно **1,0** — ви порівнюєте матрицю саму із собою;
* якщо **більше 2** — найімовірніше взято автомобільну мережу з односторонніми
  вулицями, або місто має річку/залізницю, яку нема де перетнути. Друге —
  цікавий результат, перше — помилка. Розберіться, що саме у вас, і напишіть це.

Якщо серед віддалей трапились `inf` — частина графа не зв'язана з рештою.
Не замінюйте `inf` великим числом мовчки: спершу з'ясуйте, які це точки.

In [20]:
pts10 = make_points(top10)
ids10, dist10 = snap(pts10)
check_snap(pts10, ids10, dist10, 'мала задача')

t0 = time.time(); D10 = road_matrix(ids10); print('матриця за %.3f с' % (time.time() - t0))
E10 = straight_matrix(pts10)
print('недосяжних пар    :', int(np.isinf(D10).sum()))
print('асиметрія         : %.0f м' % float(np.max(np.abs(D10 - D10.T))))
print('коефіцієнт обходу : %.2f' % detour_factor(D10, E10))

мала задача: точок 11, різних вузлів 11
   відстань точка -> вузол: медіана 12.1 м, максимум 81.3 м
   усі точки на різних вузлах: добре
матриця за 0.109 с
недосяжних пар    : 0
асиметрія         : 0 м
коефіцієнт обходу : 1.20


> **Асиметрія.** $D_{ij} \\neq D_{ji}$ — не помилка: односторонній рух робить
> шлях туди і назад різним. На пішохідній мережі асиметрія мала, на
> автомобільній — помітна. Подивіться на своє число і поясніть його у звіті.

---
## 6. Задача комівояжера

| метод | що робить | складність |
|---|---|---|
| найближчий сусід | щоразу йде в найближчу невідвідану точку | $O(n^2)$ |
| 2-opt | розвертає ділянки готового маршруту, поки коротшає | $O(n^2)$ за прохід |
| Held–Karp | динамічне програмування за підмножинами — **точний** | $O(2^n n^2)$ часу, $O(2^n n)$ пам'яті |

Held–Karp дано готовим: він не є метою роботи, але без нього ви не дізнаєтесь,
**наскільки** помиляються перші два.

### Завдання 6.1 — реалізуйте перші два методи

In [ ]:
def tour_length(D, tour):
    return float(sum(D[tour[i], tour[i + 1]] for i in range(len(tour) - 1)))

def nearest_neighbour(D):
    """Маршрут 0 -> ... -> 0. Щоразу йдемо в найближчу невідвідану точку."""
    k = len(D)
    tour, left = [0], set(range(1, k))
    while left:
        now = tour[-1]
        nxt = min(left, key=lambda j: D[now, j])   
        tour.append(nxt)
        left.remove(nxt)
    return tour + [0]

def two_opt(D, tour):
    """Беремо маршрут і розвертаємо ділянки, поки довжина зменшується."""
    best = tour[:]
    best_len = tour_length(D, best)
    improved = True
    while improved:
        improved = False
        for i in range(1, len(best) - 2):         
            for j in range(i + 1, len(best) - 1):
                new = best[:i] + best[i:j + 1][::-1] + best[j + 1:]   
                L = tour_length(D, new)
                if L < best_len - 1e-9:
                    best, best_len, improved = new, L, True
    return best

In [22]:
def held_karp(D):
    """Точний розв'язок. Вузол 0 — старт і фініш. ДАНО, писати не треба."""
    k = len(D); N = k - 1; INF = float('inf')
    dp = [[INF] * N for _ in range(1 << N)]
    par = [[-1] * N for _ in range(1 << N)]
    for j in range(N):
        dp[1 << j][j] = D[0, j + 1]
    for S in range(1 << N):
        for j in range(N):
            cur = dp[S][j]
            if cur == INF or not (S >> j) & 1:
                continue
            for t in range(N):
                if (S >> t) & 1:
                    continue
                S2 = S | (1 << t); v = cur + D[j + 1, t + 1]
                if v < dp[S2][t]:
                    dp[S2][t] = v; par[S2][t] = j
    full = (1 << N) - 1
    bj = min(range(N), key=lambda j: dp[full][j] + D[j + 1, 0])
    best = dp[full][bj] + D[bj + 1, 0]
    path = []; S = full; j = bj
    while j != -1:
        path.append(j + 1); pj = par[S][j]; S ^= (1 << j); j = pj
    return [0] + path[::-1] + [0], float(best)

### Самоперевірка

Ці твердження мають виконатись для **будь-якого** міста. Якщо хоч одне впало —
шукайте помилку в коді, а не в даних.

In [23]:
def check_tour(tour, k, label):
    assert tour[0] == 0 and tour[-1] == 0, '%s: маршрут має починатись і закінчуватись на старті' % label
    assert sorted(tour[:-1]) == list(range(k)), '%s: кожна точка рівно один раз' % label
    print('%s: структура маршруту правильна (%d точок)' % (label, k))

nn10 = nearest_neighbour(D10)
op10 = two_opt(D10, nn10)
ex10, L_opt = held_karp(D10)

for t, lbl in [(nn10, 'найближчий сусід'), (op10, '2-opt'), (ex10, 'Held-Karp')]:
    check_tour(t, len(ids10), lbl)

assert tour_length(D10, op10) <= tour_length(D10, nn10) + 1e-6, '2-opt не може бути гіршим за вхідний маршрут'
assert L_opt <= tour_length(D10, op10) + 1e-6, 'точний розв\u2019язок не може бути гіршим за наближений'
print()
print('усі перевірки пройдено')

найближчий сусід: структура маршруту правильна (11 точок)
2-opt: структура маршруту правильна (11 точок)
Held-Karp: структура маршруту правильна (11 точок)

усі перевірки пройдено


Вивід: три рядки «структура маршруту правильна (11 точок)» і «усі перевірки пройдено». Перевіряється, що маршрут починається й закінчується на старті, кожна точка є рівно один раз, а також дві логічні умови: 2-opt не гірший за свій вхід, а точний розв'язок не гірший за наближений. Якби якась умова впала, це помилка в коді, не в даних.

In [24]:
rows = []
for name, tour in [('найближчий сусід', nn10), ('2-opt', op10), ('Held-Karp (точний)', ex10)]:
    L = tour_length(D10, tour)
    rows.append({'метод': name, 'довжина, м': round(L),
                 'гірше оптимуму, %': round(100 * (L / L_opt - 1), 2)})
display(pd.DataFrame(rows).set_index('метод'))

print()
print('оптимальний порядок:')
for step, i in enumerate(ex10):
    print('  %2d. %s' % (step, pts10['name'].iloc[i]))
print()
print('довжина: %.2f км' % (L_opt / 1000))

,"довжина, м","гірше оптимуму, %"
метод,,
найближчий сусід,18149,10.96
2-opt,16396,0.24
Held-Karp (точний),16357,0.00



оптимальний порядок:
   0. СТАРТ: Чернівці
   1. Franz Josef
   2. Музей Буковинської Діаспори
   3. Музей етнографії та стародавньої історії
   4. Пам'ятник Сидору Воробкевичу
   5. Музей пожежно-рятувальної справи
   6. Художній музей
   7. Музей Авіації та Космонавтики
   8. Музей Володимира Івасюка
   9. Турецька криниця
  10. Костел Воздвиження Святого Христа
  11. СТАРТ: Чернівці

довжина: 16.36 км


Результат завдання 6.1 — малa задача

На 10 об'єктах точний метод Held–Karp дає 16,36 км. Найближчий сусід на 10,96% довший, бо жадібно йде в найближчу точку й наприкінці змушений повертатися з далекої. 2-opt, який розвертає ділянки маршруту, гірший за оптимум лише на 0,24% (39 м). Висновок: проста евристика з локальним покращенням на малій задачі майже досягає точного результату, а всі самоперевірки пройдені.

### Завдання 6.2 — велика задача

Спершу оцініть, чи можливий тут точний розв'язок. Порахуйте для
$n = 11, 16, 21, 26, 31$ розмір таблиці Held–Karp ($2^{n-1}\\cdot(n-1)$ клітин)
і скільки це гігабайтів при 8 байтах на клітину.

In [ ]:
rows = []
for n in [11, 16, 21, 26, 31]:
    cells = 2 ** (n - 1) * (n - 1)         
    rows.append({'n': n, 'клітин': cells, 'ГБ (8 байт/клітину)': round(cells * 8 / 1e9, 4)})
display(pd.DataFrame(rows).set_index('n'))

,клітин,ГБ (8 байт/клітину)
n,,
11,10240,0.0001
16,491520,0.0039
21,20971520,0.1678
26,838860800,6.7109
31,32212254720,257.6980


Результат завдання 6.2 — чому точний метод недоступний

Held–Karp зберігає таблицю з 2^(n−1)·(n−1) клітин, тож кожна нова точка подвоює її. Для 11 точок це 10 тисяч клітин, для 21 — 0,17 ГБ, для 26 — 6,7 ГБ, а для 31 (старт і 30 пам'яток) — понад 32 мільярди клітин, близько 258 ГБ. Це набагато більше за оперативну пам'ять звичайного комп'ютера, тому точний розв'язок на 30 точках недоступний через обсяг пам'яті, а не просто через час.

Подивіться на останній рядок і напишіть у звіті, **чому саме** точний розв'язок
на 30 точках недоступний. Слово «довго» відповіддю не є.

In [26]:
pts30 = make_points(top_score)
ids30, dist30 = snap(pts30)
check_snap(pts30, ids30, dist30, 'велика задача')

D30 = road_matrix(ids30)
E30 = straight_matrix(pts30)
print('коефіцієнт обходу: %.2f' % detour_factor(D30, E30))

nn30 = nearest_neighbour(D30)
t0 = time.time(); op30 = two_opt(D30, nn30); t_op30 = time.time() - t0
check_tour(op30, len(ids30), '2-opt на 30 точках')

L_nn30, L_op30 = tour_length(D30, nn30), tour_length(D30, op30)
print()
print('найближчий сусід : %8.0f м' % L_nn30)
print('2-opt            : %8.0f м  за %.3f с  (коротше на %.1f%%)'
      % (L_op30, t_op30, 100 * (1 - L_op30 / L_nn30)))
print()
print('ЦЕ — база порівняння для лабораторної роботи № 8.')

велика задача: точок 31, різних вузлів 31
   відстань точка -> вузол: медіана 16.0 м, максимум 85.0 м
   усі точки на різних вузлах: добре
коефіцієнт обходу: 1.21
2-opt на 30 точках: структура маршруту правильна (31 точок)

найближчий сусід :    33182 м
2-opt            :    29485 м  за 0.005 с  (коротше на 11.1%)

ЦЕ — база порівняння для лабораторної роботи № 8.


Результат великої задачі (30 об'єктів)

Для 31 точки (старт і 30 об'єктів) найближчий сусід дав 33,2 км, а 2-opt — 29,5 км за 0,013 с, тобто на 11,1% коротше. Оптимум невідомий, тому 29,5 км — це лише верхня оцінка.

---
## 7. Просторова індексація: H3

H3 — ієрархічна сітка шестикутників. Рівень 9 — приблизно 200 м по ребру.

📖 <https://h3geo.org/docs/core-library/restable/>

### Завдання 7.1

Розкладіть **усіх** кандидатів по комірках H3 рівня 9 і побудуйте картограму
щільності. На тій самій карті покажіть межу міста і точки вашого маршруту.

In [ ]:
import h3
from shapely.geometry import Polygon

H3_RES = 9
print('середнє ребро комірки: %.0f м' % h3.average_hexagon_edge_length(H3_RES, unit='m'))

rp = cand.geometry.representative_point()
cand['h3'] = [h3.latlng_to_cell(p.y, p.x, H3_RES) for p in rp]

dens = cand['h3'].value_counts()

hexes = gpd.GeoDataFrame(
    {'h3': dens.index, 'n': dens.values},
    geometry=[Polygon([(lng, lat) for lat, lng in h3.cell_to_boundary(c)]) for c in dens.index],
    crs='EPSG:4326')

import branca.colormap as bcm
cmap = bcm.linear.YlOrRd_09.scale(1, int(dens.max()))
pts_h3 = make_points(top10)

c0 = boundary.geometry.iloc[0].centroid
fmap_h3 = folium.Map(location=[c0.y, c0.x], zoom_start=13, tiles=BASEMAP)
folium.GeoJson(boundary.geometry.iloc[0].__geo_interface__,
               style_function=lambda f: {'fill': False, 'color': '#333', 'weight': 2}).add_to(fmap_h3)
folium.GeoJson(hexes.to_json(),
               style_function=lambda f: {'fillColor': cmap(f['properties']['n']), 'color': '#999',
                                         'weight': 0.5, 'fillOpacity': 0.65},
               tooltip=folium.GeoJsonTooltip(fields=['n'], aliases=['об\'єктів:'])).add_to(fmap_h3)
for k, r in pts_h3.iterrows():
    folium.CircleMarker([r.geometry.y, r.geometry.x], radius=5, color='#0b3d91',
                        fill=True, fill_opacity=1, tooltip=r['name']).add_to(fmap_h3)
fmap_h3.save('lr5_h3.html')

cum = dens.sort_values(ascending=False).cumsum()
k_half = int((cum < len(cand) / 2).sum() + 1)
print('кандидатів: %d, комірок з ними: %d' % (len(cand), len(dens)))
print('половина пам\'яток (%d) зосереджена у %d комірках з %d' % (len(cand) / 2, k_half, len(dens)))
print('найщільніша комірка: %d об\'єктів' % dens.max())
fmap_h3

середнє ребро комірки: 201 м
кандидатів: 71, комірок з ними: 40
половина пам'яток (35) зосереджена у 10 комірках з 40
найщільніша комірка: 6 об'єктів


Я розклала 71 кандидата по шестикутниках H3 рівня 9 (ребро близько 200 м). Об'єкти потрапили у 40 комірок, причому половина пам'яток (35) лежить лише у 10 із них, а в найщільнішій комірці їх 6. Тобто пам'ятки сконцентровані в центрі міста. Для маршруту це означає щільне ядро з кількох сусідніх комірок і кілька віддалених точок, які розтягують його довжину.

> **Пастка.** `h3.cell_to_boundary` повертає пари **(lat, lng)**, а
> `shapely.Polygon` чекає **(x, y)**, тобто (lng, lat). Переплутаєте — місто
> опиниться в Індійському океані. Це видно одразу, тому дивіться на карту.

### Питання до звіту

У скількох комірках зосереджено половину всіх пам'яток вашого міста?
Що це говорить про форму маршруту?

---
## 8. Просторове поєднання

### Завдання 8.1

Для кожної точки маршруту знайдіть найближчий заклад із шару `amenities`
(`sjoin_nearest`) і виведіть таблицю з відстанню в метрах.

Нагадування: `sjoin_nearest` рахує в одиницях CRS. У градусах результат буде
безглуздим — перепроєктуйте обидва шари в метричну систему.

In [ ]:
am = gpd.GeoDataFrame(
    {'заклад': amenities['name'], 'тип': amenities['amenity']},
    geometry=amenities.geometry.representative_point(), crs=amenities.crs).to_crs(32635)
am = am[am['заклад'].notna()]            

pm = pts10.to_crs(32635)[['name', 'geometry']]

near = gpd.sjoin_nearest(pm, am, how='left', distance_col='відстань_м')
near = near[~near.index.duplicated(keep='first')]       
near['відстань_м'] = near['відстань_м'].round(0)
display(near[['name', 'заклад', 'тип', 'відстань_м']].reset_index(drop=True))
print('медіана відстані до найближчого закладу: %.0f м' % near['відстань_м'].median())

,name,заклад,тип,відстань_м
0,СТАРТ: Чернівці,У Петровича,cafe,90.0
1,Костел Воздвиження Святого Христа,Гауптвахта,cafe,24.0
2,Музей Володимира Івасюка,О.С.А.,pharmacy,282.0
3,Музей пожежно-рятувальної справи,Долина Плюс питна вода,drinking_water,57.0
4,Музей Буковинської Діаспори,Георг Палац,restaurant,555.0
5,Franz Josef,Кайзер,restaurant,597.0
6,Пам'ятник Сидору Воробкевичу,Rita Steinberg,restaurant,185.0
7,Турецька криниця,La Multi ani,restaurant,130.0
8,Художній музей,Літературне кафе,cafe,10.0
9,Музей етнографії та стародавньої історії,Rita Steinberg,restaurant,255.0


медіана відстані до найближчого закладу: 130 м


Для кожної точки маршруту я знайшла найближчий заклад (кафе, ресторан, питна вода тощо) через `sjoin_nearest`, у метричній проєкції. Медіана відстані — 130 м, тобто біля більшості пам'яток є де зупинитися за пару хвилин ходьби. Найдалі від закладів (555 і 597 м) Музей Буковинської Діаспори та Franz Josef. Відстані рахуються по прямій, а не вулицями, тому реальний шлях трохи довший.

---
## 9. Маршрут на карті

Лінія маршруту — це **геометрія ребер графа**, а не відрізки між пам'ятками.

### Завдання 9.1

Побудуйте карту, на якій **обидві** лінії: справжній маршрут по ребрах і
прямі відрізки між точками. Різниця має бути видно оком.

In [29]:
node_xy = nodes.set_index('osmid')[['x', 'y']]

def route_coords(tour, ids):
    """Координати маршруту по ребрах графа у форматі (lat, lon) для folium."""
    coords = []
    for a, b in zip(tour[:-1], tour[1:]):
        try:
            path = nx.shortest_path(G, ids[a], ids[b], weight='length')
        except nx.NetworkXNoPath:
            path = [ids[a], ids[b]]
        for nd in path:
            coords.append((float(node_xy.loc[nd, 'y']), float(node_xy.loc[nd, 'x'])))
    return coords

def straight_coords(tour, pts):
    """Ті самі точки, але прямі відрізки."""
    return [(pts.geometry.y.iloc[i], pts.geometry.x.iloc[i]) for i in tour]

def add_route(fmap, tour, ids, pts, title, color, show):
    grp = folium.FeatureGroup(name=title, show=show)
    folium.PolyLine(route_coords(tour, ids), color=color, weight=5, opacity=0.9,
                    tooltip='по ребрах графа').add_to(grp)
    folium.PolyLine(straight_coords(tour, pts), color='#888', weight=2, dash_array='6',
                    opacity=0.9, tooltip='прямі відрізки').add_to(grp)
    for step, i in enumerate(tour[:-1]):
        label = 'С' if step == 0 else str(step)
        folium.Marker([pts.geometry.y.iloc[i], pts.geometry.x.iloc[i]], tooltip=str(pts['name'].iloc[i]),
                      icon=folium.DivIcon(html='<div style="background:%s;color:#fff;border-radius:50%%;'
                                               'width:22px;height:22px;line-height:22px;text-align:center;'
                                               'font-size:12px;font-weight:bold">%s</div>' % (color, label),
                                          icon_size=(22, 22), icon_anchor=(11, 11))).add_to(grp)
    grp.add_to(fmap)

c0 = pts10.geometry.iloc[0]
fmap = folium.Map(location=[c0.y, c0.x], zoom_start=14, tiles=BASEMAP)
add_route(fmap, ex10, ids10, pts10, 'точний маршрут, 10 об\'єктів', '#c0392b', True)
add_route(fmap, op30, ids30, pts30, '2-opt, 30 об\'єктів', '#1f5fbf', False)
folium.LayerControl(collapsed=False).add_to(fmap)
fmap.save('lr5_маршрут.html')

L_roads = tour_length(D10, ex10)
L_lines = tour_length(E10, ex10)
print('маршрут із 10 об\'єктів: по вулицях %.2f км, по прямих %.2f км (вулицями довше у %.2f раза)'
      % (L_roads / 1000, L_lines / 1000, L_roads / L_lines))
fmap

маршрут із 10 об'єктів: по вулицях 16.36 км, по прямих 12.93 км (вулицями довше у 1.27 раза)


На карті показані два варіанти того самого маршруту: по ребрах пішохідного графа і прямими відрізками між точками. Вулицями маршрут із 10 об'єктів складає 16,36 км, по прямих — 12,93 км, тобто вулицями довше в 1,27 раза. Різницю видно оком: пряма лінія проходить крізь квартали, а реальний шлях їх огинає. Додатково на карті є шар із маршрутом 2-opt для 30 об'єктів.

---
## 10. Результати для лабораторної роботи № 8

Збережіть файл, який ЛР8 прочитає і спробує побити.

In [30]:
result = {
    'студент': STUDENT_NAME,
    'номер': LIST_NUMBER,
    'місто': CITY,
    'мала_задача': {
        'точок': len(ids10),
        'оптимум_м': round(L_opt, 1),
        'оптимальний_порядок': [str(pts10['name'].iloc[i]) for i in ex10],
        'найближчий_сусід_м': round(tour_length(D10, nn10), 1),
        'двоопт_м': round(tour_length(D10, op10), 1),
    },
    'велика_задача': {
        'точок': len(ids30),
        'найближчий_сусід_м': round(L_nn30, 1),
        'двоопт_м': round(L_op30, 1),
        'двоопт_с': round(t_op30, 4),
        'точки': [{'name': str(pts30['name'].iloc[i]),
                   'lat': float(pts30.geometry.y.iloc[i]),
                   'lon': float(pts30.geometry.x.iloc[i])} for i in range(len(pts30))],
        'матриця_віддалей_м': [[None if not np.isfinite(v) else round(float(v), 1)
                                for v in row] for row in D30],
    },
}
with open('lr5_результати.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=1)
print('збережено lr5_результати.json (%.1f КБ)' % (os.path.getsize('lr5_результати.json') / 1024))

збережено lr5_результати.json (16.0 КБ)


---
## 11. Звіт

Відповіді пишіть просто в наступній комірці, числами зі своїх обчислень.

1. **Скільки об'єктів ви відкинули** на кроці 2 і за яким правилом? Чи погодились
   би ви віддати цей фільтр замовнику без пояснень?
2. **Які сигнали ввійшли у ваш рейтинг і з якими вагами?** Чому саме так?
3. **Скільки об'єктів збіглося** у ваших двох рейтингах із десяти? Що це
   означає для слова «найкращі»?
4. **Який у вас коефіцієнт обходу** і чи типовий він? Якщо ні — що в місті
   його пояснює?
5. **На скільки відсотків** найближчий сусід гірший за точний розв'язок?
   Чи здивувало вас це число?
6. **Чому точний розв'язок на 30 точках недоступний?** Назвіть конкретну
   величину, яка цьому заважає.
7. **Що у ваших даних виявилось неправильним** — дублікати, пропущені назви,
   об'єкти не на місці? Як ви з цим вчинили?

### Декларація про використання штучного інтелекту

Зазначте, чи користувались ШІ-помічником, у яких саме місцях і як перевіряли
отриманий код. Використання дозволене; **неперевірене** використання — ні.

In [33]:
REPORT = """
1. Відкинуто 22 з 93 об'єктів шару heritage. Правило: лишаю об'єкт, якщо в нього є тег historic або tourism зі списку
   (museum, attraction, gallery, artwork, viewpoint), є назва name і це не дрібний знак (boundary_stone, wayside_cross,
   wayside_shrine, milestone, survey_point). За тегами підійшли всі 93 (запит уже відбирав за тегами), без назви було 21
   (7 меморіалів, 5 attraction, 4 залізничні вагони та ін.), ще 1 об'єкт відкинуто як дрібний знак. Фільтр я б не віддала
   замовнику без пояснень: правило «без назви» викидає й справжні об'єкти, а список дрібних знаків я склала сама, і він
   помилився (див. п. 7).

2. score = 0.35*wiki + 0.30*rank(мови) + 0.15*музей/галерея + 0.10*historic + 0.10*rank(площа будівлі).
   wiki (0.35) — найбільша вага: стаття у Wikipedia/Wikidata означає, що об'єкт визнав хтось сторонній; її мають 7 з 71
   кандидатів, і всі 7 потрапили в топ-10. Мови (0.30): назву перекладають тим, що шукають іноземці; переклад (не
   рахуючи name:uk) мають лише 19 з 71. Музей/галерея (0.15): такі місця призначені для відвідування (13 з 71).
   historic (0.10) слабший сигнал: він є у 40 з 71, з них 32 — меморіали. Площа (0.10) — найслабший сигнал, бо великим
   буває і склад. Ваги я підібрала з міркувань, не оптимізувала, тож вони можуть бути іншими.

3. Збіглося 4 з 10: Костел Воздвиження Святого Христа, Художній музей, Музей етнографії та стародавньої історії,
   Музей Авіації та Космонавтики. Решта 6 різні, тобто «найкращі» залежить від критерію: площа ставить вгору
   «Німецький будинок» (він у списку двічі — дублікат), «Колишню єврейську лікарню» й пам'ятник митрополиту, а мій
   рейтинг — музеї зі статтями (Івасюка, Буковинської Діаспори). Застереження: площа є лише у 19 з 71 об'єктів
   (решта — точки поза будівлею), тож порівняння чесне тільки для них.

4. Коефіцієнт обходу 1.20 (10 точок + старт) і 1.21 (30 точок + старт). Для пішохідної мережі типово 1.2–1.6, тобто
   це нижня межа, нічого дивного. Асиметрія 0 м: на пішохідній мережі немає односторонніх вулиць. Маршрут із 10 об'єктів
   вулицями 16.36 км, по прямих 12.93 км (у 1.27 раза довше). Найбільший обхід має пара Franz Josef — Музей Буковинської
   Діаспори: по прямій 289 м, вулицями 1087 м (у 3.77 раза). Причину цього обходу я не перевіряла.

5. Найближчий сусід гірший за точний розв'язок на 10.96% (18149 м проти 16357 м). 2-opt гірший лише на 0.24% (16396 м),
   тобто точний розв'язок виграє у 2-opt усього 39 м із 16.4 км. Мене здивувало, наскільки добре 2-opt наближається до
   оптимуму. На 30 точках 2-opt коротший за найближчого сусіда на 11.1% (29485 м проти 33182 м).

6. Точний метод Held-Karp зберігає таблицю з 2^(n-1)*(n-1) клітин. Для n=11 це 10 240 клітин, для n=21 — 20 971 520
   (0.17 ГБ), для n=26 — 838 860 800 (6.7 ГБ), для n=31 (старт + 30 пам'яток) — 32 212 254 720 клітин, це приблизно
   257.7 ГБ при 8 байтах на клітину. Заважає розмір пам'яті: кожна додаткова точка подвоює таблицю (2 у степені n).

7. Що виявилось неправильним у даних. (а) 21 з 93 об'єктів без назви — відкинула. (б) Дублікати: «Німецький будинок»
   двічі на відстані 4.9 м; «Steinbarg» і «Вуличне мистецтво» — 45 м; «Костел Воздвиження» і «600 років моєму рідному
   місту» — 54 м. У кожній парі лишила об'єкт із вищим балом; у всіх кандидатах так видалено 12 з 71 (лишилось 59), у
   топ-30 змінилося 3 об'єкти. (в) Не пам'ятки в рейтингу: «Стадіон», «Центральна площа», «Паротяг 159-495»,
   «Вид на місто Чернівці» (теги attraction/viewpoint ставлять як завгодно). (г) Мій фільтр дрібних знаків помилково
   відкинув «Церкву Покрови» (вона має тег wayside_shrine). (д) Назви іншою мовою чи неповні: «Franz Josef»,
   «Chernivitsi Ghetto of 1941», «Gedenkstätte für Genozid…». (е) Із 71 кандидата 32 — меморіали, а площу будівлі мають
   лише 19.

Додатковий висновок (його не було в завданні). Критерій «цінності» не враховує відстань, тому дві далекі точки —
Музей Авіації та Космонавтики і Музей Володимира Івасюка — розтягують маршрут: ділянки до них і між ними (4.6 км і
4.4 км) становлять 55% з 16.36 км. Якщо важливіша зручність маршруту, а не «цінність», рейтинг треба штрафувати за
відстань від центру.

Декларація про використання ШІ:

Користувалась ШІ-помічником Claude як допоміжним інструментом. Структуру розв'язку, вибір критерію рейтингу, ваги сигналів, поріг для дублікатів (60 м) і висновки у звіті визначала я сама.

ШІ використовувала у трьох місцях:
1. Підказки щодо синтаксису й роботи бібліотек (geopandas, osmnx, networkx, folium, h3) під час написання коду.
2. Перевірка коду: ШІ переглядав мої комірки, вказував на помилки й пропонував виправлення. Після першого прогону на 
справжніх даних таким чином знайшли й виправили три проблеми: порядок при однакових балах залежав від комп'ютера, площа 
майже не працювала (3 з 71 об'єктів), мови рахували name:uk.
3. Пояснення та підготовка до захисту.

Як перевіряла: повністю перезапустила ноутбук (Restart and Run All) на своєму комп'ютері; звірила числа у звіті з виводами; перевірила контрольні ознаки з завдання (різних вузлів стільки ж, скільки точок; коефіцієнт обходу 1,2 у межах 1,2–1,6; найближчий сусід ≥ 2-opt ≥ Held-Karp). Кожну комірку можу пояснити.
"""
print(REPORT)


1. Відкинуто 22 з 93 об'єктів шару heritage. Правило: лишаю об'єкт, якщо в нього є тег historic або tourism зі списку
   (museum, attraction, gallery, artwork, viewpoint), є назва name і це не дрібний знак (boundary_stone, wayside_cross,
   wayside_shrine, milestone, survey_point). За тегами підійшли всі 93 (запит уже відбирав за тегами), без назви було 21
   (7 меморіалів, 5 attraction, 4 залізничні вагони та ін.), ще 1 об'єкт відкинуто як дрібний знак. Фільтр я б не віддала
   замовнику без пояснень: правило «без назви» викидає й справжні об'єкти, а список дрібних знаків я склала сама, і він
   помилився (див. п. 7).

2. score = 0.35*wiki + 0.30*rank(мови) + 0.15*музей/галерея + 0.10*historic + 0.10*rank(площа будівлі).
   wiki (0.35) — найбільша вага: стаття у Wikipedia/Wikidata означає, що об'єкт визнав хтось сторонній; її мають 7 з 71
   кандидатів, і всі 7 потрапили в топ-10. Мови (0.30): назву перекладають тим, що шукають іноземці; переклад (не
   рахуючи name:uk) мають лише 19